# Task 4.3 — Atmosphere Model Comparison

**Classical Mechanics · Project I · EETAC, UPC**

Compares the ISA multi-layer model against the simplified single-exponential model ρ(z) = ρ₀·exp(−z/H).
Answers: maximum altitude difference, drag evolution, and physical realism.

In [ ]:
# ── Download simulation dependencies from GitHub ─────────────────────────────
import urllib.request, os
BASE = "https://raw.githubusercontent.com/gdlcjoel-lab/PROYECTOMECANICA/main/Definitivos/"
for f in ["rocket_simulation.py", "plot_utils.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(BASE + f, f)
        print(f"Downloaded {f}")

# Use non-interactive backend so plots render inline in Colab
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
%matplotlib inline
print("Setup complete ✓")

In [ ]:
import numpy as np
from rocket_simulation import (simulate_rocket, compute_metrics, print_metrics,
                               density_isa, density_exponential, RHO0, H_SCALE)

DENSITY_CASES = [density_isa, density_exponential]
LABELS        = ["ISA (multi-layer)", "Exponential ρ₀·exp(−z/H)"]
COLORS        = ["#0D2B55", "#B07D2A"]
LINESTYLES    = ["--", "-"]

print("=" * 60)
print("  TASK 4.3 — ATMOSPHERE MODEL COMPARISON")
print("=" * 60)

results, metrics_list = [], []
for density_func, label in zip(DENSITY_CASES, LABELS):
    print(f"\n>>> {label}")
    t, r, v, drag, mass, accel = simulate_rocket(density_func=density_func)
    m_obj = compute_metrics(t, r, v)
    results.append((t, r, v, drag, mass, accel))
    metrics_list.append(m_obj)
    print_metrics(m_obj, label)

m_isa, m_exp = metrics_list
delta_z = m_exp["z_max"] - m_isa["z_max"]

print("\n1. COMPARE MAXIMUM ALTITUDE:")
print(f"   ISA:  {m_isa['z_max']/1000:.3f} km")
print(f"   Exp:  {m_exp['z_max']/1000:.3f} km")
print(f"   Δ  = {delta_z/1000:+.3f} km")

print("\n2. DENSITY COMPARISON AT KEY ALTITUDES:")
for alt_km in [0, 2, 5, 8, 11, 15, 20]:
    ri = density_isa(alt_km*1000); re = density_exponential(alt_km*1000)
    print(f"   {alt_km:2d} km  ISA={ri:.3e}  Exp={re:.3e}  ratio={re/ri:.3f}")

print("\n3. WHICH MODEL IS MORE REALISTIC?")
print("   ISA — captures tropopause kink at 11 km, matches measured data to <1% up to 85 km.")

## Plots
Altitude, drag, speed and air density experienced along trajectory.

In [ ]:
t0, r0, v0, d0 = results[0][0], results[0][1], results[0][2], results[0][3]
t1, r1, v1, d1 = results[1][0], results[1][1], results[1][2], results[1][3]

rho_isa_arr = np.array([density_isa(max(float(z),0)) for z in r0[:,2]])
rho_exp_arr = np.array([density_exponential(max(float(z),0)) for z in r1[:,2]])

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle("Task 4.3 — ISA vs Exponential Atmosphere", fontsize=14, fontweight="bold", color="#0D2B55")
fig.patch.set_facecolor("white")
axes = axes.flatten()

data = [
    (t0, r0[:,2]/1000, t1, r1[:,2]/1000, "Altitude vs Time", "Time [s]", "Altitude [km]"),
    (t0, d0, t1, d1, "Aerodynamic Drag vs Time", "Time [s]", "Drag [N]"),
    (t0, np.linalg.norm(v0,axis=1), t1, np.linalg.norm(v1,axis=1), "Speed vs Time", "Time [s]", "Speed [m/s]"),
    (t0, rho_isa_arr, t1, rho_exp_arr, "Air Density Experienced", "Time [s]", "ρ [kg/m³]"),
]
for ax, (x0,y0,x1,y1,title,xl,yl) in zip(axes, data):
    ax.plot(x0, y0, color="#0D2B55", linestyle="--", linewidth=2, label=LABELS[0])
    ax.plot(x1, y1, color="#B07D2A", linestyle="-",  linewidth=2, label=LABELS[1])
    ax.set_title(title, fontweight="bold", color="#0D2B55")
    ax.set_xlabel(xl); ax.set_ylabel(yl)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig("task_4_3_plots.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved as task_4_3_plots.png")